# 🏥 Health Condition Prediction: End-to-End Ensemble Pipeline

### Multiclass Risk Classification with XGBoost, CatBoost, and LightGBM
---
#### 📌 Problem Overview
This project develops an end-to-end Machine Learning pipeline to predict patient health conditions (**Fit**, **At-Risk**, **Unhealthy**) from demographic, physiological, and behavioral markers.

#### 🔑 Key Pipeline Highlights
- **Dataset:** Kaggle Playground Series S6E7 (690,088 training records, 295,753 test records)
- **Domain Feature Engineering:** WHO BMI categories, Calorie-to-Step efficiency, Active-to-Sleep balance, Cardio-Metabolic risk index, and Composite lifestyle scores
- **Validation Strategy:** 5-Fold Stratified Cross-Validation with Balanced Accuracy metric
- **Models:** XGBoost, CatBoost, and LightGBM
- **Ensemble:** Out-of-Fold Soft-Voting with probability calibration
- **Explainability:** SHAP TreeExplainer for clinical interpretability

## 1. Environment Setup & Library Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & Cross-Validation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Gradient Boosted Decision Tree Algorithms
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import shap

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
print('All libraries loaded successfully!')

## 2. Exploratory Data Analysis & Target Distribution

In [ ]:
# Load training and testing datasets
if os.path.exists('/kaggle/input/competitions/playground-series-s6e7/train.csv'):
    train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')
elif os.path.exists('../data/sample_patients.csv'):
    train_df = pd.read_csv('../data/sample_patients.csv')
    test_df = train_df.drop(columns=['health_condition']).head(500)
else:
    from src.data_loader import generate_synthetic_patients
    train_df = generate_synthetic_patients(3000)
    test_df = train_df.drop(columns=['health_condition']).head(500)

print(f'Training shape: {train_df.shape}')
train_df.head()

In [ ]:
# Target Class Distribution Visualizations
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
order = ['fit', 'at-risk', 'unhealthy']
sns.countplot(data=train_df, x='health_condition', order=order, ax=ax[0], palette=['#31C48D', '#F59E0B', '#EF4444'])
ax[0].set_title('Target Class Frequency Counts', fontweight='bold')

train_df['health_condition'].value_counts().plot.pie(
    autopct='%1.1f%%', ax=ax[1], colors=['#F59E0B', '#EF4444', '#31C48D'], startangle=90
)
ax[1].set_ylabel('')
ax[1].set_title('Target Class Proportions', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Domain-Driven Feature Engineering

We create domain-specific physiological interactions and lifestyle indexes:
- **`bmi_category`**: WHO classification bins (<18.5, 18.5-24.9, 25-29.9, ≥30)
- **`calorie_per_step`**: Metabolic energy efficiency indicator
- **`active_to_sleep_ratio`**: Physical exertion vs. restorative sleep balance
- **`hydration_index`**: Fluid intake normalized by metabolic expenditure
- **`cardio_metabolic_risk`**: Normalized interaction between resting heart rate and BMI
- **`lifestyle_score`**: Composite score across diet, activity, stress, sleep quality, and smoking/alcohol

In [ ]:
def engineer_features(df):
    df = df.copy()
    # 1. BMI Category
    bmi = pd.to_numeric(df['bmi'], errors='coerce').fillna(24.0)
    cond = [bmi < 18.5, (bmi >= 18.5) & (bmi < 25.0), (bmi >= 25.0) & (bmi < 30.0), bmi >= 30.0]
    df['bmi_category'] = np.select(cond, ['underweight', 'normal', 'overweight', 'obese'], default='normal')
    
    # 2. Calorie per step
    df['calorie_per_step'] = df['calorie_expenditure'].fillna(2000) / (df['step_count'].fillna(5000) + 100)
    
    # 3. Active to sleep ratio
    df['active_to_sleep_ratio'] = (df['exercise_duration'].fillna(30)/60.0) / (df['sleep_duration'].fillna(7) + 0.1)
    
    # 4. Hydration index
    df['hydration_index'] = df['water_intake'].fillna(2.0) / ((df['calorie_expenditure'].fillna(2000)/1000.0) + 0.5)
    
    # 5. Cardio-metabolic risk
    df['cardio_metabolic_risk'] = (df['heart_rate'].fillna(72)/70.0) * (df['bmi'].fillna(24)/22.0)
    
    # 6. Composite lifestyle score
    lifestyle = np.zeros(len(df))
    lifestyle += df['stress_level'].astype(str).str.lower().map({'low': 2.0, 'moderate': 1.0, 'high': 0.0}).fillna(1.0)
    lifestyle += df['sleep_quality'].astype(str).str.lower().map({'good': 2.0, 'average': 1.0, 'poor': 0.0}).fillna(1.0)
    lifestyle += df['physical_activity_level'].astype(str).str.lower().map({'active': 2.0, 'moderate': 1.0, 'sedentary': 0.0}).fillna(1.0)
    lifestyle += df['smoking_alcohol'].astype(str).str.lower().map({'no': 2.0, 'occasional': 1.0, 'yes': 0.0}).fillna(1.0)
    lifestyle += df['diet_type'].astype(str).str.lower().map({'vegan': 1.5, 'veg': 1.0, 'non-veg': 0.5}).fillna(0.5)
    df['lifestyle_score'] = lifestyle
    return df

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)
print(f'Total features after domain engineering: {train_fe.shape[1]}')

## 4. Preprocessing & Leakage-Free Pipeline Setup

In [ ]:
num_cols = [
    'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count',
    'exercise_duration', 'water_intake', 'calorie_per_step', 'active_to_sleep_ratio',
    'hydration_index', 'cardio_metabolic_risk', 'lifestyle_score'
]
cat_cols = [
    'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level',
    'smoking_alcohol', 'gender', 'bmi_category'
]

X = train_fe.drop(columns=[c for c in ['id', 'health_condition'] if c in train_fe.columns])
y_raw = train_fe['health_condition']
X_test = test_fe.drop(columns=[c for c in ['id', 'health_condition'] if c in test_fe.columns])

label_enc = LabelEncoder()
label_enc.fit(['at-risk', 'fit', 'unhealthy'])
y = label_enc.transform(y_raw)

# ColumnTransformer Preprocessor
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

X_trans = preprocessor.fit_transform(X)
X_test_trans = preprocessor.transform(X_test)
print(f'Transformed feature matrix shape: {X_trans.shape}')

## 5. 5-Fold Stratified Cross-Validation & Model Training

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
num_classes = 3

oof_xgb = np.zeros((len(X_trans), num_classes))
oof_cat = np.zeros((len(X_trans), num_classes))
oof_lgb = np.zeros((len(X_trans), num_classes))

test_pred_xgb = np.zeros((len(X_test_trans), num_classes))
test_pred_cat = np.zeros((len(X_test_trans), num_classes))
test_pred_lgb = np.zeros((len(X_test_trans), num_classes))

xgb_scores, cat_scores, lgb_scores = [], [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trans, y), 1):
    print(f'\n--- FOLD {fold}/5 ---')
    X_train, y_train = X_trans[train_idx], y[train_idx]
    X_val, y_val = X_trans[val_idx], y[val_idx]
    
    # 1. XGBoost
    xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='mlogloss')
    xgb.fit(X_train, y_train)
    oof_xgb[val_idx] = xgb.predict_proba(X_val)
    s_xgb = balanced_accuracy_score(y_val, np.argmax(oof_xgb[val_idx], axis=1))
    xgb_scores.append(s_xgb)
    test_pred_xgb += xgb.predict_proba(X_test_trans) / 5.0
    print(f'  XGBoost Balanced Accuracy:  {s_xgb:.4f}')
    
    # 2. CatBoost
    cat = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=7, verbose=0, random_seed=42)
    cat.fit(X_train, y_train)
    oof_cat[val_idx] = cat.predict_proba(X_val)
    s_cat = balanced_accuracy_score(y_val, np.argmax(oof_cat[val_idx], axis=1))
    cat_scores.append(s_cat)
    test_pred_cat += cat.predict_proba(X_test_trans) / 5.0
    print(f'  CatBoost Balanced Accuracy: {s_cat:.4f}')
    
    # 3. LightGBM
    lgb = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, num_leaves=63, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
    lgb.fit(X_train, y_train)
    oof_lgb[val_idx] = lgb.predict_proba(X_val)
    s_lgb = balanced_accuracy_score(y_val, np.argmax(oof_lgb[val_idx], axis=1))
    lgb_scores.append(s_lgb)
    test_pred_lgb += lgb.predict_proba(X_test_trans) / 5.0
    print(f'  LightGBM Balanced Accuracy: {s_lgb:.4f}')

## 6. Out-of-Fold Soft Voting Ensemble & Validation Evaluation

In [ ]:
oof_ensemble = (oof_xgb + oof_cat + oof_lgb) / 3.0
oof_preds = np.argmax(oof_ensemble, axis=1)

print('=' * 55)
print('MODEL BENCHMARKS (5-Fold Stratified Cross-Validation)')
print('=' * 55)
print(f'XGBoost Mean CV Balanced Accuracy:  {np.mean(xgb_scores):.4f}')
print(f'CatBoost Mean CV Balanced Accuracy: {np.mean(cat_scores):.4f}')
print(f'LightGBM Mean CV Balanced Accuracy: {np.mean(lgb_scores):.4f}')
ensemble_acc = balanced_accuracy_score(y, oof_preds)
print(f'\nFINAL SOFT-VOTING ENSEMBLE BALANCED ACCURACY: {ensemble_acc:.4f}')

print('\nClassification Report:')
print(classification_report(y, oof_preds, target_names=['at-risk', 'fit', 'unhealthy']))

In [ ]:
# Out-of-Fold Confusion Matrix
cm = confusion_matrix(y, oof_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['at-risk', 'fit', 'unhealthy'], yticklabels=['at-risk', 'fit', 'unhealthy'])
plt.title('Out-of-Fold Ensemble Confusion Matrix', fontweight='bold', pad=12)
plt.xlabel('Predicted Condition')
plt.ylabel('True Condition')
plt.tight_layout()
plt.show()

## 7. Model Interpretability with SHAP

In [ ]:
all_feature_names = num_cols + cat_cols
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_trans[:200])

plt.title('SHAP Feature Importance (Class 0: At-Risk)', fontweight='bold')
shap.summary_plot(shap_values[0] if isinstance(shap_values, list) else shap_values, X_trans[:200], feature_names=all_feature_names, show=False)
plt.tight_layout()
plt.show()

## 8. Final Test Submission Generation

In [ ]:
test_ensemble = (test_pred_xgb + test_pred_cat + test_pred_lgb) / 3.0
final_test_preds = np.argmax(test_ensemble, axis=1)
final_labels = label_enc.inverse_transform(final_test_preds)

sub_id = test_df['id'] if 'id' in test_df.columns else np.arange(len(test_df))
submission = pd.DataFrame({
    'id': sub_id,
    'health_condition': final_labels
})

submission.to_csv('submission.csv', index=False)
print(f'Submission generated successfully with {len(submission)} rows!')
submission.head()